# Objetivos

¿El sentimiento expresado por escrito coincide con el promedio de **4.5/5 estrellas** del negocio? Exploraremos las 500 reseñas y compararemos sus ratings con las predicciones de un modelo fijo; el resultado es una señal exploratoria, no una evaluación definitiva del servicio.

In [1]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../data/raw/reviews.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/reviews.csv")
reviews = pd.read_csv(DATA_PATH)

print(f"Archivo: {DATA_PATH.resolve()}")
print(f"Forma: {reviews.shape[0]} filas x {reviews.shape[1]} columnas")
print("\nTipos de datos:")
print(reviews.dtypes)
print("\nValores faltantes:")
print(reviews.isna().sum())

print("\nDistribución de ratings:")
rating_counts = reviews["rating"].value_counts(dropna=False).sort_index()
rating_summary = pd.DataFrame({"count": rating_counts, "percent": rating_counts / len(reviews) * 100})
print(rating_summary.round(1))
print(f"Promedio de rating: {reviews['rating'].mean():.2f} / 5")
print(f"Ratings fuera del rango 1-5: {(~reviews['rating'].between(1, 5)).sum()}")

text_lengths = reviews["review_text"].fillna("").astype(str).str.len()
print("\nLongitud de reseñas (caracteres):")
print(text_lengths.describe().round(1))
print(f"Textos vacíos o faltantes: {(text_lengths == 0).sum()}")
print(f"IDs duplicados: {reviews['review_id'].duplicated().sum()}")
print(f"Textos exactos duplicados: {reviews['review_text'].duplicated().sum()}")

Archivo: /workspaces/machine-learning-python-template-Miguel-Contreras/data/raw/reviews.csv
Forma: 500 filas x 3 columnas

Tipos de datos:
review_id      int64
rating         int64
review_text      str
dtype: object

Valores faltantes:
review_id      0
rating         0
review_text    0
dtype: int64

Distribución de ratings:
        count  percent
rating                
1          12      2.4
2          18      3.6
3          38      7.6
4          72     14.4
5         360     72.0
Promedio de rating: 4.50 / 5
Ratings fuera del rango 1-5: 0

Longitud de reseñas (caracteres):
count    500.0
mean     170.2
std       27.1
min       86.0
25%      155.0
50%      173.0
75%      189.0
max      230.0
Name: review_text, dtype: float64
Textos vacíos o faltantes: 0
IDs duplicados: 0
Textos exactos duplicados: 7


El perfil identifica la distribución y los posibles problemas estructurales. Ahora comparamos una pequeña muestra de textos con sus ratings, sin automatizar la clasificación del sentimiento.

In [2]:
sample_by_rating = (
    reviews.groupby("rating", group_keys=False)
    .sample(n=2, random_state=42)
    .sort_values("rating")
)
print(sample_by_rating[["rating", "review_text"]].to_string(index=False, max_colwidth=240))

 rating                                                                                                                                                                              review_text
      1       Visited Harbor House Café last weekend. The coffee tasted burnt. The cashier was rude when i asked about the menu. The place felt cramped and chaotic. Not sure I'll be returning.
      1 Regular customer at Harbor House Café here. The coffee tasted burnt. The staff seemed annoyed to be there. It was way too loud to hold a conversation. Disappointing experience overall.
      2               Grabbed a quick coffee at Harbor House Café this morning. The food was bland and overpriced. Our server got our order wrong twice. Probably won't recommend it to friends.
      2                                             Regular customer at Harbor House Café here. The coffee tasted burnt. Nobody checked on us the entire meal. Disappointing experience overall.
      3                            

## EDA: insights y limpieza

- El promedio es **4.50/5**; el 72% da 5 estrellas y solo el 6% da 1–2. La distribución está inclinada hacia ratings altos.
- Hay 500 reseñas completas, sin ratings fuera de 1–5 ni IDs repetidos. Los textos tienen una longitud media de 170 caracteres (rango: 86–230).
- La muestra por rating refleja extremos mayormente coherentes y experiencias mixtas en 3–4 estrellas; es orientativa, no representativa por sí sola.
- Hay 7 textos exactos repetidos (493 textos únicos). Revisarlos, pero no eliminarlos sin confirmar que sean duplicados de captura; no se justifica otra limpieza previa.

## Plan de acción y modelo

Compararemos las bandas de sentimiento con el promedio de estrellas y revisaremos manualmente 20 reseñas y cada discrepancia crítica. Usaremos una sola instancia de `nlptown/bert-base-multilingual-uncased-sentiment`, que predice estrellas 1–5, y el mapeo fijo **1–2 negativo, 3 neutral, 4–5 positivo**. Sus predicciones son una señal exploratoria: el modelo no está calibrado específicamente para Harbor House Café.

In [3]:
from transformers import pipeline

MODEL_ID = "nlptown/bert-base-multilingual-uncased-sentiment"
sentiment_model = pipeline(
    "sentiment-analysis",
    model=MODEL_ID,
    tokenizer=MODEL_ID,
    device=-1,
)
raw_predictions = sentiment_model(
    reviews["review_text"].tolist(),
    batch_size=16,
    truncation=True,
)

reviews["predicted_stars"] = [int(result["label"].split()[0]) for result in raw_predictions]
reviews["prediction_confidence"] = [result["score"] for result in raw_predictions]
reviews["sentiment_band"] = pd.cut(
    reviews["predicted_stars"],
    bins=[0, 2, 3, 5],
    labels=["negative", "neutral", "positive"],
).astype("string")

band_order = ["positive", "neutral", "negative"]
band_percent = reviews["sentiment_band"].value_counts(normalize=True).reindex(band_order, fill_value=0) * 100
print("Reseñas clasificadas:", len(reviews))
print("Rating humano promedio:", round(reviews["rating"].mean(), 2), "/ 5")
print("Estrellas predichas promedio:", round(reviews["predicted_stars"].mean(), 2), "/ 5")
print("Porcentaje por banda (%):")
print(band_percent.round(1).to_string())

false_negatives = reviews.loc[
    (reviews["predicted_stars"] <= 2) & (reviews["rating"] >= 4),
    ["review_id", "rating", "predicted_stars", "sentiment_band", "prediction_confidence", "review_text"],
]
print("\nFalsos negativos (rating humano 4-5, predicción 1-2):", len(false_negatives))
print(false_negatives.to_string(index=False, max_colwidth=220))

manual_sample = (
    reviews.groupby("rating", group_keys=False)
    .sample(n=4, random_state=42)
    .sort_values(["rating", "review_id"])
)
print("\nMuestra manual estratificada (4 por rating):")
print(
    manual_sample[["review_id", "rating", "predicted_stars", "sentiment_band", "prediction_confidence", "review_text"]]
    .to_string(index=False, max_colwidth=220)
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Reseñas clasificadas: 500
Rating humano promedio: 4.5 / 5
Estrellas predichas promedio: 4.4 / 5
Porcentaje por banda (%):
sentiment_band
positive    82.4
neutral      8.2
negative     9.4

Falsos negativos (rating humano 4-5, predicción 1-2): 16
 review_id  rating  predicted_stars sentiment_band  prediction_confidence                                                                                                                                                                                                   review_text
        14       4                1       negative               0.444348                                  Tried Harbor House Café after seeing it recommended online. Every dish was bursting with flavor. The team clearly cares about the customers. Already planning my next visit.
        15       5                1       negative               0.389345                                                                                               Every dish was bursting wi

Las predicciones ya están asociadas a cada reseña en `reviews`. A continuación comparamos la proporción de cada banda con ratings humanos agrupados usando el mismo mapeo.

In [4]:
human_band = pd.cut(
    reviews["rating"],
    bins=[0, 2, 3, 5],
    labels=["negative", "neutral", "positive"],
).astype("string")
human_percent = human_band.value_counts(normalize=True).reindex(band_order, fill_value=0) * 100

band_comparison = pd.DataFrame(
    {
        "human rating share (%)": human_percent,
        "model sentiment share (%)": band_percent,
    }
)
band_comparison["model - human (pp)"] = (
    band_comparison["model sentiment share (%)"] - band_comparison["human rating share (%)"]
)
print("Rating-based comparison using the fixed 1-2 / 3 / 4-5 mapping:")
print(band_comparison.round(1).to_string())
print(
    "\nMean-star difference (model - business):",
    round(reviews["predicted_stars"].mean() - reviews["rating"].mean(), 2),
)
print("Review-level prediction columns:", ["predicted_stars", "sentiment_band", "prediction_confidence"])

Rating-based comparison using the fixed 1-2 / 3 / 4-5 mapping:
          human rating share (%)  model sentiment share (%)  model - human (pp)
positive                    86.4                       82.4                -4.0
neutral                      7.6                        8.2                 0.6
negative                     6.0                        9.4                 3.4

Mean-star difference (model - business): -0.1
Review-level prediction columns: ['predicted_stars', 'sentiment_band', 'prediction_confidence']


## Resultados, auditoría y conclusión

El modelo predice **82.4% positivo, 8.2% neutral y 9.4% negativo**, frente al mapeo de ratings humanos (**86.4%, 7.6% y 6.0%**). La diferencia es de **−4.0, +0.6 y +3.4 puntos porcentuales**; el promedio de estrellas predicho es **4.4**, cercano al **4.5** del negocio. La mayor discrepancia está en la cola negativa, no en el promedio.

Hay **16 falsos negativos**: ratings humanos 4–5 con predicción 1–2 (3.2% del total). Por ejemplo, el ID 15 tiene 5 estrellas y recomienda volver, pero recibe 1; el ID 14 elogia la comida, la atención y una próxima visita, pero recibe 1. Varios casos repiten expresiones favorables como “Every dish was bursting with flavor” o intención de volver. En algunos se menciona una crítica puntual del servicio, aunque el rating general es alto; revisar el contexto antes de tratar la predicción como una señal negativa real.

La muestra manual estratificada incluye 20 reseñas. Las bandas coinciden con el rating agrupado en 19 de 20: los ejemplos de 1–2 y 4–5 estrellas se clasifican en la banda esperada; el ID 480 (rating 3) cae en negativa pese a una redacción mixta, por lo que es un posible exceso de negatividad. El ID 166 (rating 4, predicción 5) es una diferencia de una estrella, aunque conserva banda positiva. Esta lectura pequeña no sustituye una validación con etiquetas humanas.

**Para la account manager:** el promedio de 4.5 estrellas y el predominio de lenguaje positivo son compatibles. No comunicar el 9.4% negativo del modelo como proporción real de clientes insatisfechos: incluye 16 reseñas de 4–5 estrellas mal clasificadas según la regla acordada. Revisar esas discrepancias y, antes de usar el modelo en decisiones, validar una muestra etiquetada por personas.